In [4]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

images_path = Path("dataset/images")
data = json.loads(Path("dataset/gt_public.json").read_text())
image_ids = list(data.keys())
n_train = int(0.8 * len(image_ids))


def make_mask(boxes):
    mask = np.zeros((256, 256), dtype=np.uint8)
    yy, xx = np.mgrid[0:256, 0:256]
    for x, y, w, h in boxes:
        cx = x + w / 2
        cy = y + h / 2
        r = max(w, h) / 2
        inside = (xx + 0.5 - cx) ** 2 + (yy + 0.5 - cy) ** 2 <= r ** 2
        mask[inside] = 1
    return mask

Using device: cuda


In [2]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
    )


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.down1 = conv_block(1, 32)
        self.down2 = conv_block(32, 64)
        self.down3 = conv_block(64, 128)
        self.bottom = conv_block(128, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = conv_block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv1 = conv_block(64, 32)
        self.out = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        b = self.bottom(self.pool(d3))
        u3 = self.conv3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.conv2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.conv1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


model = UNet().to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0], device=device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

batch_size = 16
for step in range(23):
    if step == 3:
        torch.cuda.synchronize()
        start = time.time()

    batch_ids = random.sample(image_ids, batch_size)
    images = torch.stack([
        torch.tensor(np.array(Image.open(images_path / f"{i}.png").convert("L")), dtype=torch.float32).unsqueeze(0) / 255.0
        for i in batch_ids
    ]).to(device)
    masks = torch.stack([
        torch.tensor(make_mask(data[i]), dtype=torch.float32).unsqueeze(0)
        for i in batch_ids
    ]).to(device)

    loss = loss_fn(model(images), masks)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
seconds_per_image = (time.time() - start) / (20 * batch_size)
print(f"U-Net: {seconds_per_image:.4f} s per image, about {seconds_per_image * n_train / 60:.1f} min per epoch")

U-Net: 0.0050 s per image, about 3.4 min per epoch


In [3]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

for min_size in [256, 800]:
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=min_size)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    model.to(device).train()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9)

    batch_size = 4
    for step in range(23):
        if step == 3:
            torch.cuda.synchronize()
            start = time.time()

        batch_ids = random.sample(image_ids, batch_size)
        images = [
            torch.tensor(np.array(Image.open(images_path / f"{i}.png").convert("RGB")), dtype=torch.float32).permute(2, 0, 1).to(device) / 255.0
            for i in batch_ids
        ]
        targets = [
            {
                "boxes": torch.tensor([[x, y, x + w, y + h] for x, y, w, h in data[i]], dtype=torch.float32).to(device),
                "labels": torch.ones(len(data[i]), dtype=torch.int64).to(device),
            }
            for i in batch_ids
        ]

        loss = sum(model(images, targets).values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    torch.cuda.synchronize()
    seconds_per_image = (time.time() - start) / (20 * batch_size)
    print(f"Faster R-CNN, size {min_size}: {seconds_per_image:.4f} s per image, about {seconds_per_image * n_train / 60:.1f} min per epoch")

Faster R-CNN, size 256: 0.0132 s per image, about 9.1 min per epoch
Faster R-CNN, size 800: 0.0241 s per image, about 16.7 min per epoch
